<a href="https://colab.research.google.com/github/VinayaSharada/KateelLearningDemosToStudents/blob/main/TreasuryAnalytics/InvoiceLevelCollectionsPrediction/invoice_level_collections_prediction.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Invoice-Level Collections Prediction

This notebook upgrades the treasury collections workflow into a classroom-ready forecast pack that predicts both **late-payment risk** and **expected payment timing** at invoice level. It is designed for synthetic training data by default and can score a real open-invoice extract without retraining the model.

## Learning goals

- Build invoice-level features for collections and cash forecasting.
- Train a classifier for late-payment likelihood and a regressor for days versus due date.
- Compare a contractual due-date forecast with an ML-adjusted expected-payment-date forecast.
- Export a collections action queue and calendar-ready treasury files.
- Interpret forecast value with human judgment, governance, and capital assumptions.

## Model-governance and classroom guardrails

- Synthetic data is used by default for classroom training.
- Uploading current open invoices **scores** them but does not retrain the model.
- A production model would require historical invoices with actual payment dates.
- Predictions have an error margin and are not payment commitments.
- Strategic-customer, dispute, and concentration decisions still require human judgment.
- The run summary includes model version, random seed, training timestamp, and test metrics.


## 1. Setup and imports


In [ ]:
import json
from datetime import datetime

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, mean_absolute_error, mean_squared_error, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

rng = np.random.default_rng(42)
pd.set_option("display.max_columns", 50)


## Step-by-Step Explanation

### What this cell is doing
1. Imports the Python libraries needed for data preparation, modeling, charting, and export generation.
2. Detects whether the notebook is running in Colab so download and upload behavior can stay classroom-friendly.
3. Sets a reproducible random number generator seed so the training example is stable across runs.

### How to interpret the result
- A clean run here means the rest of the notebook has the tools it needs.
- The `IN_COLAB` flag matters because upload and download steps behave differently in browser notebooks versus local Jupyter.
- The fixed seed makes the classroom discussion repeatable; if the seed changes, metrics and sample invoices may shift.


## 2. Configuration cell


In [ ]:
CURRENCY_CODE = "INR"
DISPLAY_SCALE = "crore"
AS_OF_DATE = None
FORECAST_HORIZONS = [7, 14, 30]
COST_OF_CAPITAL_RATE = 0.09
USE_SYNTHETIC_DATA = True
RANDOM_SEED = 42

MODEL_VERSION = "invoice-collections-predictor-v2"
TRAINING_TIMESTAMP = datetime.utcnow().isoformat() + "Z"
AS_OF = pd.Timestamp(AS_OF_DATE).normalize() if AS_OF_DATE else pd.Timestamp.today().normalize()
rng = np.random.default_rng(RANDOM_SEED)

def format_amount(value, currency=CURRENCY_CODE, scale=DISPLAY_SCALE):
    value = float(value)
    if currency == "INR":
        scale_map = {
            "crore": (10_000_000, "crore"),
            "lakh": (100_000, "lakh"),
            "rupee": (1, "INR"),
        }
        divisor, label = scale_map.get(scale, (1, "INR"))
        return f"₹{value / divisor:,.2f} {label}" if divisor != 1 else f"₹{value:,.0f}"
    return f"{currency} {value:,.0f}"

def money_axis(ax):
    if CURRENCY_CODE == "INR" and DISPLAY_SCALE == "crore":
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _pos: f"₹{x / 10_000_000:,.2f} Cr"))
    elif CURRENCY_CODE == "INR" and DISPLAY_SCALE == "lakh":
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _pos: f"₹{x / 100_000:,.2f} L"))
    else:
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _pos: f"{CURRENCY_CODE} {x:,.0f}"))

print({
    "CURRENCY_CODE": CURRENCY_CODE,
    "DISPLAY_SCALE": DISPLAY_SCALE,
    "AS_OF": str(AS_OF.date()),
    "FORECAST_HORIZONS": FORECAST_HORIZONS,
    "COST_OF_CAPITAL_RATE": COST_OF_CAPITAL_RATE,
    "USE_SYNTHETIC_DATA": USE_SYNTHETIC_DATA,
    "RANDOM_SEED": RANDOM_SEED,
})


## Step-by-Step Explanation

### What this cell is doing
1. Defines the business assumptions that should be easy for faculty or students to change before running the workflow.
2. Sets the model version, run timestamp, and as-of date used for scoring and forecast windows.
3. Builds formatting helpers so the notebook can present INR in crore or lakh instead of unexplained dollar figures.

### How to interpret the result
- This is the control panel for the notebook. If the teaching context changes, this is the first place to edit.
- `AS_OF` drives the timing logic for forecasts, slippages, and escalation windows.
- `COST_OF_CAPITAL_RATE` affects the economic-value estimate, so students should treat it as an explicit assumption rather than a hidden fact.


## 3. Template and synthetic-data scaffolding


In [ ]:
TEMPLATE_COLUMNS = [
    "invoice_id", "customer_id", "customer_name", "industry", "region", "channel",
    "invoice_date", "due_date", "invoice_amount", "payment_terms",
    "avg_days_beyond_terms", "payment_history_score", "open_dispute",
    "relationship_strength", "seasonality_stress"
]

template_rows = [
    ["INV-1001", "CUST-0001", "Meridian Foods Pvt Ltd", "Retail", "North", "Distributor", "2026-05-01", "2026-05-31", 4200000, 30, 12, 82, 0, "Stable", "Low"],
    ["INV-1002", "CUST-0002", "Alcott Manufacturing", "Manufacturing", "West", "Field Sales", "2026-05-04", "2026-06-18", 12500000, 45, 28, 61, 1, "Weak", "High"],
    ["INV-1003", "CUST-0003", "BrightPath Diagnostics", "Healthcare", "South", "Email", "2026-05-10", "2026-06-09", 3100000, 30, 5, 91, 0, "Strategic", "Low"],
]

template_df = pd.DataFrame(template_rows, columns=TEMPLATE_COLUMNS)
TEMPLATE_PATH = "invoice_data_template.csv"
template_df.to_csv(TEMPLATE_PATH, index=False)
print(f"Wrote {TEMPLATE_PATH} for optional scoring-only uploads.")

if IN_COLAB:
    colab_files.download(TEMPLATE_PATH)
else:
    print(f"Running outside Colab — find the template at: {TEMPLATE_PATH}")

def generate_customer_master(n_customers=220):
    industries = ["Technology", "Retail", "Manufacturing", "Healthcare", "Chemicals", "Consumer"]
    regions = ["North", "South", "East", "West"]
    channels = ["Email", "Portal", "Distributor", "Field Sales"]
    relationships = ["Strategic", "Stable", "Weak"]
    seasonality_levels = ["Low", "Medium", "High"]

    customers = []
    for i in range(n_customers):
        customer_id = f"CUST-{i+1:04d}"
        industry = rng.choice(industries)
        region = rng.choice(regions)
        relationship = rng.choice(relationships, p=[0.2, 0.55, 0.25])
        channel = rng.choice(channels)
        seasonality = rng.choice(seasonality_levels, p=[0.45, 0.35, 0.20])
        history_score = float(np.clip(rng.normal(75, 15), 30, 98))
        avg_days = float(np.clip(rng.normal(14, 12), 0, 75))
        customers.append({
            "customer_id": customer_id,
            "customer_name": f"Synthetic Customer {i+1:04d}",
            "industry": industry,
            "region": region,
            "channel": channel,
            "relationship_strength": relationship,
            "seasonality_stress": seasonality,
            "payment_history_score": history_score,
            "avg_days_beyond_terms": avg_days,
        })
    return pd.DataFrame(customers)

def generate_invoices(customers, n_invoices, labeled=True, as_of=AS_OF):
    invoices = []
    for i in range(n_invoices):
        customer = customers.sample(1, random_state=int(rng.integers(0, 1_000_000))).iloc[0]
        invoice_date = as_of - pd.Timedelta(days=int(rng.integers(15, 180)))
        payment_terms = int(rng.choice([15, 30, 45, 60], p=[0.1, 0.5, 0.25, 0.15]))
        due_date = invoice_date + pd.Timedelta(days=payment_terms)
        invoice_amount = float(np.round(rng.lognormal(mean=15.4, sigma=0.75), 2))
        open_dispute = int(rng.random() < (0.22 if customer["relationship_strength"] == "Weak" else 0.08))

        latent_days = (
            0.35 * customer["avg_days_beyond_terms"]
            + 12 * open_dispute
            + (7 if customer["seasonality_stress"] == "High" else 2 if customer["seasonality_stress"] == "Medium" else 0)
            + (8 if customer["relationship_strength"] == "Weak" else -3 if customer["relationship_strength"] == "Strategic" else 0)
            + max(0, 85 - customer["payment_history_score"]) * 0.3
            + rng.normal(0, 4)
        )
        predicted_days = int(np.round(latent_days))
        record = {
            "invoice_id": f"INV-{i+1:05d}",
            "customer_id": customer["customer_id"],
            "customer_name": customer["customer_name"],
            "industry": customer["industry"],
            "region": customer["region"],
            "channel": customer["channel"],
            "relationship_strength": customer["relationship_strength"],
            "seasonality_stress": customer["seasonality_stress"],
            "payment_history_score": float(customer["payment_history_score"]),
            "avg_days_beyond_terms": float(customer["avg_days_beyond_terms"]),
            "invoice_date": invoice_date.normalize(),
            "due_date": due_date.normalize(),
            "payment_terms": payment_terms,
            "invoice_amount": invoice_amount,
            "open_dispute": open_dispute,
        }
        if labeled:
            record["actual_days_vs_due"] = predicted_days
            record["actual_payment_date"] = due_date.normalize() + pd.Timedelta(days=predicted_days)
            record["paid_late"] = int(predicted_days > 0)
        invoices.append(record)
    return pd.DataFrame(invoices)


## Step-by-Step Explanation

### What this cell is doing
1. Creates the upload template for real open-invoice scoring so students know the expected schema.
2. Builds a synthetic customer master and invoice generator for classroom-safe model training and scoring.
3. Encodes realistic collections drivers such as disputes, relationship quality, seasonality, and payment behavior into the synthetic data.

### How to interpret the result
- The template tells students which fields are mandatory versus optional for scoring.
- Synthetic data is deliberately structured so the model can learn meaningful treasury patterns without exposing confidential invoices.
- The latent payment-delay logic is the hidden business process the model is trying to approximate from observable features.


## 4. Build the training set and the scoring set


In [ ]:
customers = generate_customer_master()
training_df = generate_invoices(customers, n_invoices=1200, labeled=True, as_of=AS_OF)

if USE_SYNTHETIC_DATA:
    scoring_df = generate_invoices(customers, n_invoices=150, labeled=False, as_of=AS_OF)
    print(f"Generated {len(scoring_df)} synthetic open invoices for scoring.")
else:
    OWN_DATA_PATH = "invoice_data_template.csv"
    if IN_COLAB:
        print("Upload your current open-invoice CSV to score (no retraining occurs):")
        uploaded = colab_files.upload()
        OWN_DATA_PATH = next(iter(uploaded)) if uploaded else OWN_DATA_PATH
    scoring_df = pd.read_csv(OWN_DATA_PATH)
    print(f"Loaded {len(scoring_df)} invoices from {OWN_DATA_PATH} for scoring only.")

for df in [training_df, scoring_df]:
    for col in ["invoice_date", "due_date"]:
        df[col] = pd.to_datetime(df[col])

for optional_col, default_value in {
    "avg_days_beyond_terms": training_df["avg_days_beyond_terms"].median(),
    "payment_history_score": training_df["payment_history_score"].median(),
    "open_dispute": 0,
    "relationship_strength": "Stable",
    "seasonality_stress": "Medium",
    "payment_terms": 30,
}.items():
    if optional_col not in scoring_df.columns:
        scoring_df[optional_col] = default_value
    scoring_df[optional_col] = scoring_df[optional_col].fillna(default_value)

print("Training rows:", len(training_df))
print("Scoring rows:", len(scoring_df))
training_df.head()


## Step-by-Step Explanation

### What this cell is doing
1. Generates the labeled historical training sample and the open-invoice scoring sample.
2. Keeps the scoring flow separate from the training flow so user uploads do not retrain the model.
3. Fills missing optional fields with conservative defaults so the notebook can still score imperfect exports.

### How to interpret the result
- The training dataset is the historical learning base; the scoring dataset is the current treasury decision set.
- If students upload their own data, this notebook is still demonstrating a scoring workflow, not a production retraining loop.
- Default-filled fields are a classroom convenience; in production they would need tighter data governance and validation.


## 5. Engineer features, train the classifier and regressor, and evaluate against a naive baseline


In [ ]:
feature_cols_num = ["payment_terms", "invoice_amount", "avg_days_beyond_terms", "payment_history_score", "open_dispute"]
feature_cols_cat = ["industry", "region", "channel", "relationship_strength", "seasonality_stress"]
feature_cols = feature_cols_num + feature_cols_cat

X = training_df[feature_cols]
y_class = training_df["paid_late"]
y_reg = training_df["actual_days_vs_due"]

X_train, X_test, y_class_train, y_class_test, y_reg_train, y_reg_test = train_test_split(
    X, y_class, y_reg, test_size=0.25, random_state=RANDOM_SEED
)

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), feature_cols_num),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), feature_cols_cat),
])

classifier = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(n_estimators=250, random_state=RANDOM_SEED, min_samples_leaf=2)),
])
regressor = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestRegressor(n_estimators=250, random_state=RANDOM_SEED, min_samples_leaf=2)),
])

classifier.fit(X_train, y_class_train)
regressor.fit(X_train, y_reg_train)

class_prob_test = classifier.predict_proba(X_test)[:, 1]
reg_train = regressor.predict(X_train)
reg_test = regressor.predict(X_test)
naive_train = np.zeros_like(y_reg_train, dtype=float)
naive_test = np.zeros_like(y_reg_test, dtype=float)

def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)

metrics_table = pd.DataFrame([
    {"split": "train", "model": "regressor", "rmse": rmse(y_reg_train, reg_train), "mae": float(mean_absolute_error(y_reg_train, reg_train))},
    {"split": "test", "model": "regressor", "rmse": rmse(y_reg_test, reg_test), "mae": float(mean_absolute_error(y_reg_test, reg_test))},
    {"split": "train", "model": "naive_due_date_baseline", "rmse": rmse(y_reg_train, naive_train), "mae": float(mean_absolute_error(y_reg_train, naive_train))},
    {"split": "test", "model": "naive_due_date_baseline", "rmse": rmse(y_reg_test, naive_test), "mae": float(mean_absolute_error(y_reg_test, naive_test))},
])

classifier_auc = float(roc_auc_score(y_class_test, class_prob_test))
print(f"Late-payment classifier ROC AUC: {classifier_auc:.3f}")
print(classification_report(y_class_test, (class_prob_test >= 0.5).astype(int)))
metrics_table


## Step-by-Step Explanation

### What this cell is doing
1. Separates the treasury features into numeric and categorical groups so they can be preprocessed correctly.
2. Trains two models: a classifier for the probability of late payment and a regressor for days versus due date.
3. Evaluates the timing model against a naive baseline that assumes every invoice pays exactly on the due date.

### How to interpret the result
- The classifier tells us how likely lateness is, but the regressor tells us when cash is likely to arrive.
- The naive baseline matters because a treasury team already has a simple forecast: trust the due date. The model only adds value if it beats that baseline.
- A large gap between train and test metrics is a warning that the model may be overfitting the synthetic history.


## 6. Visualize timing accuracy


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, (y_true, y_pred, title) in zip(axes, [
    (y_reg_train, reg_train, "Training set"),
    (y_reg_test, reg_test, "Test set"),
]):
    ax.scatter(y_true, y_pred, alpha=0.45, color="#0f766e")
    low = min(y_true.min(), y_pred.min())
    high = max(y_true.max(), y_pred.max())
    ax.plot([low, high], [low, high], linestyle="--", color="#dc2626")
    ax.set_title(title)
    ax.set_xlabel("Actual days vs due")
    ax.set_ylabel("Predicted days vs due")
plt.tight_layout()
plt.show()


## Step-by-Step Explanation

### What this cell is doing
1. Plots predicted versus actual days versus due date for both the training and test sets.
2. Adds a 45-degree reference line so students can see where perfect predictions would land.
3. Turns a metric table into a visual discussion about dispersion, bias, and reliability.

### How to interpret the result
- Tight clustering around the diagonal line means the timing model is tracking reality reasonably well.
- A wide spread or systematic tilt suggests the model is missing important treasury drivers.
- The test-set chart is the one to trust more for classroom discussion about generalization.


## 7. Score open invoices and build the collections action queue


In [ ]:
scoring_features = scoring_df[feature_cols].copy()
scoring_df["late_risk_probability"] = classifier.predict_proba(scoring_features)[:, 1]
scoring_df["predicted_days_vs_due"] = np.round(regressor.predict(scoring_features)).astype(int)
scoring_df["expected_payment_date"] = scoring_df["due_date"] + pd.to_timedelta(scoring_df["predicted_days_vs_due"], unit="D")

conditions = [
    (scoring_df["late_risk_probability"] >= 0.80) | (scoring_df["predicted_days_vs_due"] >= 20),
    (scoring_df["late_risk_probability"] >= 0.60) | (scoring_df["predicted_days_vs_due"] >= 10),
    (scoring_df["late_risk_probability"] >= 0.35) | (scoring_df["predicted_days_vs_due"] >= 3),
]
choices = ["Immediate action", "Escalate this week", "Monitor closely"]
scoring_df["priority_band"] = np.select(conditions, choices, default="Routine")

action_map = {
    "Immediate action": "Call customer, validate dispute status, and escalate internally",
    "Escalate this week": "Schedule follow-up and confirm expected payment timing",
    "Monitor closely": "Send reminder and track daily forecast movement",
    "Routine": "Keep in standard collections cadence",
}
owner_map = {"North": "Collections North", "South": "Collections South", "East": "Collections East", "West": "Collections West"}
scoring_df["recommended_action"] = scoring_df["priority_band"].map(action_map)
scoring_df["assigned_owner"] = scoring_df["region"].map(owner_map).fillna("Collections Shared Services")
scoring_df["escalation_date"] = np.where(
    scoring_df["priority_band"].isin(["Immediate action", "Escalate this week"]),
    (scoring_df["expected_payment_date"] - pd.Timedelta(days=3)).dt.strftime("%Y-%m-%d"),
    "",
)
scoring_df["approval_required"] = (scoring_df["invoice_amount"] >= scoring_df["invoice_amount"].quantile(0.85)) | (scoring_df["priority_band"] == "Immediate action")

prediction_cols = [
    "invoice_id", "customer_id", "customer_name", "invoice_amount", "due_date",
    "late_risk_probability", "predicted_days_vs_due", "expected_payment_date",
    "priority_band", "recommended_action"
]
action_queue_cols = prediction_cols + ["assigned_owner", "escalation_date", "approval_required"]

invoice_predictions = scoring_df[prediction_cols].sort_values(["late_risk_probability", "invoice_amount"], ascending=[False, False]).copy()
collections_action_queue = scoring_df[action_queue_cols].sort_values(["priority_band", "late_risk_probability", "invoice_amount"], ascending=[True, False, False]).copy()
invoice_predictions.head(15)


## Step-by-Step Explanation

### What this cell is doing
1. Scores each open invoice with both late-payment probability and predicted days versus due date.
2. Explicitly calculates `expected_payment_date = due_date + predicted_days_vs_due`.
3. Converts model output into an operational action queue with owners, escalation dates, and approval flags.

### How to interpret the result
- The priority band is the operational bridge between model output and treasury action.
- An invoice can matter because it is high-risk, because it is large, or because both risk and amount are high.
- `approval_required` is a governance cue: some actions should not happen automatically even if the model is confident.


## 8. Forecast baseline versus predicted inflows and identify treasury stress points


In [ ]:
daily_inflow_due_date_baseline = (
    scoring_df.groupby(scoring_df["due_date"].dt.normalize())
    .agg(expected_inflow=("invoice_amount", "sum"), invoice_count=("invoice_id", "count"))
    .reset_index()
)
daily_inflow_due_date_baseline.columns = ["date", "expected_inflow", "invoice_count"]
daily_inflow_due_date_baseline["forecast_type"] = "due_date_baseline"

daily_inflow_predicted = (
    scoring_df.groupby(scoring_df["expected_payment_date"].dt.normalize())
    .agg(expected_inflow=("invoice_amount", "sum"), invoice_count=("invoice_id", "count"))
    .reset_index()
)
daily_inflow_predicted.columns = ["date", "expected_inflow", "invoice_count"]
daily_inflow_predicted["forecast_type"] = "predicted_payment_date"

forecast_summary = []
for horizon in FORECAST_HORIZONS:
    horizon_end = AS_OF + pd.Timedelta(days=horizon)
    baseline_total = daily_inflow_due_date_baseline.loc[daily_inflow_due_date_baseline["date"].between(AS_OF, horizon_end), "expected_inflow"].sum()
    predicted_total = daily_inflow_predicted.loc[daily_inflow_predicted["date"].between(AS_OF, horizon_end), "expected_inflow"].sum()
    forecast_summary.append({
        "horizon_days": horizon,
        "baseline_inflow": baseline_total,
        "predicted_inflow": predicted_total,
        "difference": predicted_total - baseline_total,
    })
forecast_summary = pd.DataFrame(forecast_summary)

slippages = scoring_df.sort_values("predicted_days_vs_due", ascending=False)[[
    "invoice_id", "customer_name", "invoice_amount", "due_date", "predicted_days_vs_due", "expected_payment_date", "priority_band"
]].head(15)

concentration_threshold = daily_inflow_predicted["expected_inflow"].quantile(0.90) if len(daily_inflow_predicted) else 0
concentration_dates = daily_inflow_predicted.loc[daily_inflow_predicted["expected_inflow"] >= concentration_threshold].sort_values("expected_inflow", ascending=False)

reg_mae_test = float(metrics_table.query("split == 'test' and model == 'regressor'")["mae"].iloc[0])
naive_mae_test = float(metrics_table.query("split == 'test' and model == 'naive_due_date_baseline'")["mae"].iloc[0])
forecast_accuracy_gain_days = max(0.0, naive_mae_test - reg_mae_test)
avg_daily_predicted = daily_inflow_predicted["expected_inflow"].mean() if len(daily_inflow_predicted) else 0
capital_buffer_estimate = forecast_accuracy_gain_days * avg_daily_predicted
economic_value_estimate = capital_buffer_estimate * COST_OF_CAPITAL_RATE

print("Near-term inflow comparison:")
print(forecast_summary)
print("\nLargest predicted payment-date slippages:")
print(slippages)
print("\nDates with excessive predicted cash concentration:")
print(concentration_dates.head(10))
print({
    "capital_buffer_estimate": format_amount(capital_buffer_estimate),
    "economic_value_estimate": format_amount(economic_value_estimate),
})


## Step-by-Step Explanation

### What this cell is doing
1. Aggregates invoice-level predictions into two daily cash forecasts: contractual due dates and ML-adjusted expected payment dates.
2. Summarizes the next 7, 14, and 30 days of inflows, highlights the largest slippages, and flags excessive concentration dates.
3. Estimates the capital-buffer and economic value of forecast improvement using the MAE gain and cost-of-capital assumption.

### How to interpret the result
- The most important question is whether the predicted forecast materially changes the treasury picture relative to the due-date baseline.
- Large slippages identify where collections timing could alter funding or buffer decisions.
- The capital and economic value numbers are discussion estimates, not contractual truths; they are sensitive to both model quality and the cost-of-capital assumption.


## 9. Visualize the two forecast shapes


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 9))

axes[0].bar(daily_inflow_due_date_baseline["date"].astype(str), daily_inflow_due_date_baseline["expected_inflow"], color="#94a3b8")
axes[0].set_title("Contractual forecast using due dates")
money_axis(axes[0])
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(daily_inflow_predicted["date"].astype(str), daily_inflow_predicted["expected_inflow"], color="#0f766e")
axes[1].set_title("ML-adjusted forecast using expected payment dates")
money_axis(axes[1])
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


## Step-by-Step Explanation

### What this cell is doing
1. Plots the contractual forecast and the ML-adjusted forecast separately so students can compare timing shape, peaks, and gaps.
2. Uses the notebook’s INR-friendly axis formatting to keep the treasury interpretation consistent with the pack standard.
3. Turns the forecast comparison into a visual cash-planning discussion rather than a table-only exercise.

### How to interpret the result
- Look for concentration shifting across dates, not just for total inflow differences.
- If the two charts look nearly identical, the model may not be adding enough value to justify added complexity.
- If they diverge sharply, the class should ask whether the model is uncovering real behavior or amplifying synthetic assumptions.


## 10. Export the treasury deliverables


In [ ]:
invoice_payment_predictions_path = "invoice_payment_predictions.csv"
daily_inflow_due_date_baseline_path = "daily_inflow_due_date_baseline.csv"
daily_inflow_predicted_path = "daily_inflow_predicted.csv"
collections_action_queue_path = "collections_action_queue.csv"
collections_calendar_import_path = "collections_calendar_import.csv"
model_run_summary_path = "model_run_summary.json"

invoice_predictions.assign(
    due_date=invoice_predictions["due_date"].dt.strftime("%Y-%m-%d"),
    expected_payment_date=invoice_predictions["expected_payment_date"].dt.strftime("%Y-%m-%d")
).to_csv(invoice_payment_predictions_path, index=False)

daily_inflow_due_date_baseline.assign(date=daily_inflow_due_date_baseline["date"].dt.strftime("%Y-%m-%d")).to_csv(daily_inflow_due_date_baseline_path, index=False)
daily_inflow_predicted.assign(date=daily_inflow_predicted["date"].dt.strftime("%Y-%m-%d")).to_csv(daily_inflow_predicted_path, index=False)
collections_action_queue.assign(
    due_date=collections_action_queue["due_date"].dt.strftime("%Y-%m-%d"),
    expected_payment_date=collections_action_queue["expected_payment_date"].dt.strftime("%Y-%m-%d")
).to_csv(collections_action_queue_path, index=False)

calendar_import = pd.DataFrame({
    "Subject": scoring_df.apply(lambda r: f"Expected payment: {r['customer_name']} ({r['invoice_id']}) {format_amount(r['invoice_amount'])}", axis=1),
    "Start Date": scoring_df["expected_payment_date"].dt.strftime("%m/%d/%Y"),
    "All Day Event": "True",
    "Description": scoring_df.apply(lambda r: f"Invoice {r['invoice_id']} | risk={r['late_risk_probability']:.0%} | priority={r['priority_band']} | action={r['recommended_action']}", axis=1),
})
calendar_import.to_csv(collections_calendar_import_path, index=False)

run_summary = {
    "model_version": MODEL_VERSION,
    "random_seed": RANDOM_SEED,
    "training_timestamp": TRAINING_TIMESTAMP,
    "as_of_date": str(AS_OF.date()),
    "currency_code": CURRENCY_CODE,
    "display_scale": DISPLAY_SCALE,
    "use_synthetic_data": USE_SYNTHETIC_DATA,
    "classifier_test_roc_auc": classifier_auc,
    "train_rmse": float(metrics_table.query("split == 'train' and model == 'regressor'")["rmse"].iloc[0]),
    "test_rmse": float(metrics_table.query("split == 'test' and model == 'regressor'")["rmse"].iloc[0]),
    "train_mae": float(metrics_table.query("split == 'train' and model == 'regressor'")["mae"].iloc[0]),
    "test_mae": float(metrics_table.query("split == 'test' and model == 'regressor'")["mae"].iloc[0]),
    "naive_test_rmse": float(metrics_table.query("split == 'test' and model == 'naive_due_date_baseline'")["rmse"].iloc[0]),
    "naive_test_mae": float(metrics_table.query("split == 'test' and model == 'naive_due_date_baseline'")["mae"].iloc[0]),
    "forecast_accuracy_gain_days": forecast_accuracy_gain_days,
    "capital_buffer_estimate": capital_buffer_estimate,
    "economic_value_estimate": economic_value_estimate,
    "forecast_horizons": FORECAST_HORIZONS,
}
with open(model_run_summary_path, "w", encoding="utf-8") as f:
    json.dump(run_summary, f, indent=2)

print("Wrote:")
for name in [
    invoice_payment_predictions_path,
    daily_inflow_due_date_baseline_path,
    daily_inflow_predicted_path,
    collections_action_queue_path,
    collections_calendar_import_path,
    model_run_summary_path,
]:
    print(" -", name)

if IN_COLAB:
    for name in [
        invoice_payment_predictions_path,
        daily_inflow_due_date_baseline_path,
        daily_inflow_predicted_path,
        collections_action_queue_path,
        collections_calendar_import_path,
        model_run_summary_path,
    ]:
        colab_files.download(name)


## Step-by-Step Explanation

### What this cell is doing
1. Writes the required treasury-pack CSV and JSON outputs with names that downstream workflows can rely on.
2. Converts date columns into export-friendly string formats for spreadsheet, calendar, and automation tools.
3. Saves a run-summary record with model version, seed, metrics, and business assumptions.

### How to interpret the result
- These files are the handoff layer from notebook analysis into operations, automation, and governance exercises.
- `model_run_summary.json` is essential because it explains how the forecast was produced and how strong the model looked in this run.
- If any required export is missing, the notebook may still be interesting analytically but it is not fully classroom-ready for the treasury pack.


## Discussion prompts

- Which features appear operationally actionable versus merely predictive?
- How different are the due-date and predicted-payment-date cash views over 7, 14, and 30 days?
- Which invoices deserve action because of timing risk, and which because of value concentration?
- If the cost of capital changed materially, would the economic-value estimate still support using the model?
- What approval steps should remain human even when the model is confident?
